# k-NN Classification

The purpose of this notebook is to classify new replays with the help of the labeled clusters from `04_cluster_labeling`.
New replays go through the same feature helpers as `02_preprocessing`, get scaled (and reduced) with the saved models from `03_data-analysis` and get the label of their nearest labeled neighbours.


**Index**

- libraries
- settings
- importing data
- helper functions
    - 01 - blocks / clicks & movement (copied from 02)
    - 02 - random windows (copied from 02)
    - 03 - mcplayhd concept (copied from 02)
    - movement & camera features (from 03)
- building dataframes
- loading models & labels
- k-NN
    - helper functions
    - 03b - movement & camera
    - 04b - mcplayhd concept
- evaluation
- export

## libraries

In [1]:
import json
import random
import joblib
import zipfile
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.neighbors import KNeighborsClassifier

## settings

In [2]:
DATA_DIR = Path("../data/")
EXPORT_DIR = DATA_DIR / "notebooks_exports"

NEW_DATA_PATH = DATA_DIR / "validation.zip" # 10000 new replays (ids 500001..510000), one json list inside the zip
REPLAY_LABELS_PATH = DATA_DIR / "replay_labels.csv"

# ---------- copied from 02_preprocessing, values must stay the same ----------
TICKS_PER_SECOND = 20 # amount of ticks (steps) per second in minecraft

MIN_WINDOW_SIZE = 60 #size for windows to be considered windows
RANDOM_WINDOW_SIZE = 60 # size for random windows in ticks
RANDOM_SEED = 18 # to make the random windows reproducable

IDLE_THRESHOLD = 0.01 # distance moved that is still considered no movement
MAX_IDLE_TICKS = 3 # consecutive ticks without movement needed for idle
MIN_BLOCKS = 5 # min amount of blocks in window to make data meaningful

CONCEPT_WINDOW_SIZE = 50 # window size in ticks for mcplayhd's approach
Y_WEIGHT = 0.5 # weight of the y distance when searching for the best window
MAX_CLICKS = 1 # clicks per tick get capped at this value
EYE_HEIGHT = 1.62 # height of the player's eyes above their position

# ---------- models & labels from 03 / 04 ----------
METHODS = ["movement_camera", "mcplayhd_concept"]
CLUSTER_MODELS_PATHS = {method: EXPORT_DIR / f"cluster_models_{method}.joblib" for method in METHODS}
# CHANGED: k-NN now predicts the cluster, the votes come from the cluster labels of 04 (was: LABELED_REPLAYS_PATHS)
CLUSTER_LABELS_PATHS = {method: EXPORT_DIR / f"cluster_labels_{method}.csv" for method in METHODS}
KNN_RESULTS_PATHS = {method: EXPORT_DIR / f"knn_results_{method}.csv" for method in METHODS}

# ---------- k-NN ----------
KNN_NEIGHBORS = 15 # TODO: try a few values

## importing data

In [3]:
# read the json straight out of the zip, no need to unpack it
with zipfile.ZipFile(NEW_DATA_PATH) as zip_file:
    json_name = zip_file.namelist()[0]
    with zip_file.open(json_name) as f:
        new_replays = json.load(f)

# same name as in 02, so the copied loops work unchanged
def iter_data():
    yield from new_replays

# sanity checks
print(f"{len(new_replays)} new replays from {json_name}")
print(f"replay structure: {new_replays[0].keys()}")
print(pd.Series([item["fastBuilderMode"] for item in new_replays]).value_counts())

10000 new replays from thesis_data_1794427250445.json
replay structure: dict_keys(['idAnonymous', 'fastBuilderMode', 'minecraftProtocolVersion', 'stepData'])
EXTRASHORT       5349
SHORT            1414
ONESTACK          958
INCLINEDSHORT     871
NORMAL            658
LONG              393
INCLINED          357
Name: count, dtype: int64


## helper functions

copypaste from `02_preprocessing` (and the last one from `03_data-analysis`).

**DON'T CHANGE**

#### 01 - blocks / clicks & movement
`movement_per_tick` and `longest_active_window` are needed by `random_window`. `replay_features` is optional, for the blocks-per-click cheat rule.

In [4]:
def movement_per_tick(steps):
    """moved distance between each tick, returns an (x-1, ) 1D array, where x is the number of input ticks"""
    positions = np.array([[ # (x, 3) array with the (x,y,z) positions
        step["playerPosition"]["x"],
        step["playerPosition"]["y"],
        step["playerPosition"]["z"]
    ] for step in steps])
    deltas = np.diff(positions, axis=0) # (x-1, 3) array with the differences between each row
    movement = np.sqrt((deltas ** 2).sum(axis=1)) # (x-1, ) 1D array with calculated distance in each row

    return movement


def longest_active_window(movement):
    """finds the longest active window in movement, returns a slice or none if too short"""
    is_idle = movement < IDLE_THRESHOLD
    keep = np.ones(len(movement), dtype=bool)
    idle_ticks_combo = 0

    for i in range(len(movement)):
        if is_idle[i]:
            idle_ticks_combo += 1
        else:
            idle_ticks_combo = 0

        if idle_ticks_combo > MAX_IDLE_TICKS:
            # drop the whole idle part
            keep[i - idle_ticks_combo + 1 : i+1] = False

    best_start, best_length = 0, 0
    current_start, current_length = 0, 0

    for i in range(len(keep)):
        if keep[i]:
            # if keep and no active window, we start new window
            if current_length == 0:
                current_start = i
            current_length += 1
            # if current window is better, we put it as best window
            if current_length > best_length:
                best_start, best_length = current_start, current_length #
        else:
            # if not keep, we drop the window by setting length to 0
            current_length = 0

    if best_length < MIN_WINDOW_SIZE:
        return None

    return slice(best_start, best_start + best_length)


def replay_features(item):
    """calculates blocks/tick and average movement speed in replay, returns as dict"""
    # drop replays that are too short
    steps = item["stepData"]
    if len(steps) < MIN_WINDOW_SIZE:
        return None

    # drop replays with no viable window
    movement = movement_per_tick(steps)
    window = longest_active_window(movement)
    if window is None:
        return None

    # count clicks/block in window
    # use [1:] because movement of ticks 0-1 belong to step 1
    clicks = np.array([step["clicks"] for step in steps[1:]])[window].sum()
    blocks = np.array([len(step["blocksChanged"]) for step in steps[1:]])[window].sum()

    # drop if too few blocks placed in window
    if blocks < MIN_BLOCKS or clicks == 0:
        return None

    return {
        "idAnonymous": item["idAnonymous"],
        "fastBuilderMode": item["fastBuilderMode"],
        "blocks_per_click": blocks / clicks,
        "avg_speed": movement[window].mean() * TICKS_PER_SECOND,
        "window_share": len(movement[window]) / len(movement),
    }

#### 02 - random windows

In [5]:
def random_window(steps):
    """create a random window within replays, returns a list of steps"""
    # drop if too short to hold a valid window
    if len(steps) < RANDOM_WINDOW_SIZE:
        return None

    # drop if no valid active window
    active = longest_active_window(movement_per_tick(steps))
    if active is None:
        return None
    active_steps = steps[1:][active] # move active steps by 1 to align with movement

    # drop if active window too small
    if len(active_steps) < RANDOM_WINDOW_SIZE:
        return None
    start = random.randint(0, len(active_steps) - RANDOM_WINDOW_SIZE)
    return active_steps[start : start + RANDOM_WINDOW_SIZE]


def window_features(item, window):
    """creates the wide data rows, returns as dict"""
    movement = movement_per_tick(window)
    yaw = np.array([step["playerPosition"]["yaw"] for step in window])
    pitch = np.array([step["playerPosition"]["pitch"] for step in window])
    ping = np.array([step["ping"] for step in window])
    yaw_change = (np.diff(yaw) + 180) % 360 - 180 # yaw isn't limited to -180 to 180, so we limit it
    pitch_change = np.diff(pitch)

    # create the columns
    row = {
        "idAnonymous": item["idAnonymous"],
        "fastBuilderMode": item["fastBuilderMode"],
        "ping_avg": ping.mean(), # average ping
        "ping_jitter": np.abs(np.diff(ping)).mean(), # average ping change between ticks
    }

    # fill data in rows
    # creates elements like movement_1, movement_2 and so on
    row.update({f"movement_{i}": value for i, value in enumerate(movement, start=1)})
    row.update({f"yaw_change_{i}": value for i, value in enumerate(yaw_change, start=1)})
    row.update({f"pitch_change_{i}": value for i, value in enumerate(pitch_change, start=1)})
    row.update({f"clicks_{i}": step["clicks"] for i, step in enumerate(window)})
    row.update({f"sneaking_{i}": int(step["sneaking"]) for i, step in enumerate(window)})
    row.update({f"blocks_changed_{i}": len(step["blocksChanged"]) for i, step in enumerate(window)})

    return row

#### 03 - mcplayhd concept

In [6]:
def last_block_tick(steps):
    """index of the last step that placed a non-AIR block, returns an int or None if no block was placed"""
    tick_block_changed = None
    for i, step in enumerate(steps):
        for block_changed in step["blocksChanged"]:
            if block_changed["material"] != "AIR":
                tick_block_changed = i

    return tick_block_changed


def best_window_start(steps, last_tick):
    """finds the start of the window that covers the most distance without going down, returns an int or None if no window fits"""
    if last_tick is None:
        return None

    best_start = None
    best_distance = -1

    for i, step in enumerate(steps):
        # exit loop if too few ticks for window size
        if last_tick - i < CONCEPT_WINDOW_SIZE:
            break

        player_start_position = step["playerPosition"]
        player_end_position = steps[i + CONCEPT_WINDOW_SIZE - 1]["playerPosition"]

        # skip window if end position below start position
        if player_start_position["y"] > player_end_position["y"]:
            continue

        dx = player_start_position["x"] - player_end_position["x"]
        dy = player_start_position["y"] - player_end_position["y"]
        dz = player_start_position["z"] - player_end_position["z"]
        weighted_distance_squared = dx ** 2 + (dy * Y_WEIGHT) ** 2 + dz ** 2

        # check window and save it if it is better
        if weighted_distance_squared > best_distance:
            best_start = i
            best_distance = weighted_distance_squared

    return best_start


def blocks_before(steps, start):
    """positions of all blocks placed before the window, returns a set of (x, y, z) tuples"""
    block_set = set()
    for step in steps[:start]:
        for block_changed in step["blocksChanged"]:
            if block_changed["material"] != "AIR":
                location = block_changed["loc"]
                block_set.add((location["x"], location["y"], location["z"]))

    return block_set


def concept_window_features(item, start, already_placed):
    """creates the wide data rows for the window starting at start, returns as dict or None if no new block was placed"""
    steps = item["stepData"]
    window = steps[start : start + CONCEPT_WINDOW_SIZE]

    x = np.array([step["playerPosition"]["x"] for step in window])
    dx = np.diff(x)
    y = np.array([step["playerPosition"]["y"] for step in window])
    dy = np.diff(y)
    z = np.array([step["playerPosition"]["z"] for step in window])
    dz = np.diff(z)
    yaw = np.array([step["playerPosition"]["yaw"] for step in window])
    yaw_change = (np.diff(yaw) + 180) % 360 - 180
    pitch = np.array([step["playerPosition"]["pitch"] for step in window])
    pitch_change = np.diff(pitch)

    row = {
        "idAnonymous": item["idAnonymous"],
        "fastBuilderMode": item["fastBuilderMode"],
    }

    row.update({f"dx_{i}": value for i, value in enumerate(dx, start=1)})
    row.update({f"dy_{i}": value for i, value in enumerate(dy, start=1)})
    row.update({f"dz_{i}": value for i, value in enumerate(dz, start=1)})
    row.update({f"yaw_change_{i}": value for i, value in enumerate(yaw_change, start=1)})
    row.update({f"pitch_change_{i}": value for i, value in enumerate(pitch_change, start=1)})
    row.update({f"sneaking_{i}": int(step["sneaking"]) for i, step in enumerate(window)})
    row.update({f"clicks_{i}": min(step["clicks"], MAX_CLICKS) for i, step in enumerate(window)})

    # remember new blocks, but only if not on location that a block was placed in before
    for i, step in enumerate(window):
        new_blocks = []
        for block_changed in step["blocksChanged"]:
            loc = block_changed["loc"]
            position = (loc["x"], loc["y"], loc["z"])
            if (block_changed["material"] != "AIR") and (position not in already_placed):
                new_blocks.append(position)
        already_placed.update(new_blocks)

        if new_blocks:
            row[f"new_block_{i}"] = 1
            block_x, block_y, block_z = new_blocks[0]
            player = step["playerPosition"]
            row[f"block_dx_{i}"] = block_x - player["x"]
            row[f"block_dy_{i}"] = block_y - (player["y"] + EYE_HEIGHT)
            row[f"block_dz_{i}"] = block_z - player["z"]
        else:
            row[f"new_block_{i}"] = 0
            row[f"block_dx_{i}"] = 0
            row[f"block_dy_{i}"] = 0
            row[f"block_dz_{i}"] = 0

    # drop window without any blocks placed
    if sum(row[f"new_block_{i}"] for i in range(CONCEPT_WINDOW_SIZE)) == 0:
        return None

    return row

#### movement & camera features
The `movement_camera` clusters weren't built on `wide_table` directly but on 3 summary features computed in `03_data-analysis` (sections 01, 02 and 03 "create dataframe").

In [7]:
def movement_camera_features(df_wide):
    """takes a wide dataframe and returns idAnonymous, fastBuilderMode, movement_avg, movement_std, turn_avg_log"""
    movement_columns = df_wide.columns.str.startswith("movement_")
    movement_array = df_wide.loc[:, movement_columns].to_numpy()
    movement_avg = movement_array.mean(axis=1)
    movement_std = movement_array.std(axis=1)

    yaw_columns = df_wide.columns.str.startswith("yaw_change_")
    pitch_columns = df_wide.columns.str.startswith("pitch_change_")
    yaw_array = df_wide.loc[:, yaw_columns].to_numpy()
    pitch_array = df_wide.loc[:, pitch_columns].to_numpy()
    turn_array = np.hypot(yaw_array, pitch_array)
    turn_avg_log = np.log1p(turn_array.mean(axis=1))

    return pd.DataFrame({
        "idAnonymous": df_wide["idAnonymous"],
        "fastBuilderMode": df_wide["fastBuilderMode"],
        "movement_avg": movement_avg,
        "movement_std": movement_std,
        "turn_avg_log": turn_avg_log,
    })

## building dataframes
The three loops of 02 merged into one pass.

In [8]:
random.seed(RANDOM_SEED)

rows_bpc, rows_wide, rows_concept = [], [], []
samples_count = 0
for item in iter_data():
    samples_count += 1
    steps = item["stepData"]

    # 01 - blocks / clicks & movement
    row = replay_features(item)
    if row is not None:
        rows_bpc.append(row)

    # 02 - random windows
    window = random_window(steps)
    if window is not None:
        rows_wide.append(window_features(item, window))

    # 03 - mcplayhd concept
    start = best_window_start(steps, last_block_tick(steps))
    if start is not None:
        row = concept_window_features(item, start, blocks_before(steps, start))
        if row is not None:
            rows_concept.append(row)

df_new_bpc = pd.DataFrame(rows_bpc)
df_new_wide = pd.DataFrame(rows_wide)
df_new_concept = pd.DataFrame(rows_concept)

# sanity checks
for name, df in [("bpc", df_new_bpc), ("wide", df_new_wide), ("concept", df_new_concept)]:
    print(f"{name}: dataframe holds {len(df)} out of {samples_count} replays")

bpc: dataframe holds 9700 out of 10000 replays
wide: dataframe holds 9750 out of 10000 replays
concept: dataframe holds 9723 out of 10000 replays


In [9]:
df_new_movement_camera = movement_camera_features(df_new_wide)

# method -> new feature dataframe, so the k-NN cells can look it up by METHOD
new_features = {
    "movement_camera": df_new_movement_camera,
    "mcplayhd_concept": df_new_concept,
}

print(df_new_movement_camera.head(3))

   idAnonymous fastBuilderMode  movement_avg  movement_std  turn_avg_log
0       500001           SHORT      0.283304      0.101013      0.988951
1       500002      EXTRASHORT      0.089456      0.043411      0.208582
2       500003      EXTRASHORT      0.165765      0.121750      1.366677


## loading models & labels

In [10]:
cluster_models = {method: joblib.load(CLUSTER_MODELS_PATHS[method]) for method in METHODS}
cluster_labels = {method: pd.read_csv(CLUSTER_LABELS_PATHS[method]) for method in METHODS}
df_replay_labels = pd.read_csv(REPLAY_LABELS_PATH)

# sanity checks
for method, results in cluster_models.items():
    first_mode = next(iter(results))
    print(f"{method}: modes {list(results)}")
    print(f"keys per mode: {list(results[first_mode].keys())}")

movement_camera: modes ['EXTRASHORT', 'SHORT', 'NORMAL', 'ONESTACK', 'INCLINED', 'LONG', 'INCLINEDSHORT']
keys per mode: ['ids', 'features', 'scaler', 'scaled', 'kmeans', 'labels', 'nearest_distances', 'is_outlier']
mcplayhd_concept: modes ['EXTRASHORT', 'SHORT', 'NORMAL', 'ONESTACK', 'INCLINED', 'LONG', 'INCLINEDSHORT']
keys per mode: ['ids', 'scaler', 'pca', 'reduced', 'kmeans', 'labels', 'nearest_distances', 'is_outlier']


## k-NN

Per method and per mode:
- **training points**: the positions saved in the joblib (`scaled` for movement_camera, `reduced` for mcplayhd_concept)
- **training labels** *(changed)*: the cluster of each training point (`labels` in the joblib). The k-NN predicts a cluster, and that cluster's votes (`technique_votes`, `suspicious_votes`) get looked up in `cluster_labels_<method>.csv`
- **new points**: the new replays of that mode, put through the *same* scaler (and pca) of that mode, never a new fit

To decide:
- keep outliers and `too_small` clusters in the training data or drop them?
- `weights="uniform"` or `"distance"`?
- what to use as confidence for a new replay (`predict_proba`? distance to the neighbours?)

#### helper functions

In [11]:
def training_data(result, position_key):
    """returns X (positions) and y (clusters) of one mode"""
    X = result[position_key]
    y = result["labels"]
    return X, y

def transform_new(result, df_new_mode):
    """puts the new replays of one mode into the same space as the training points"""
    features = df_new_mode[result["scaler"].feature_names_in_]
    points = result["scaler"].transform(features)
    if "pca" in result:
        points = result["pca"].transform(points)
    return points

def add_cluster_votes(df_knn, df_cluster_labels):
    """joins voters, technique_votes and suspicious_votes of the predicted cluster onto each new replay"""
    vote_columns = ["mode", "cluster", "voters", "technique_votes", "suspicious_votes"]
    return df_knn.merge(df_cluster_labels[vote_columns], on=["mode", "cluster"], how="left")


### 03b - movement & camera

In [12]:
METHOD = "movement_camera"
POSITION_KEY = "scaled"

knn_frames = []
for mode, result in cluster_models[METHOD].items():
    X_train, y_train = training_data(result, POSITION_KEY)

    knn = KNeighborsClassifier(n_neighbors=KNN_NEIGHBORS)
    knn.fit(X_train, y_train)

    df_new_mode = new_features[METHOD][new_features[METHOD]["fastBuilderMode"] == mode]
    points = transform_new(result, df_new_mode)

    knn_frames.append(pd.DataFrame({
        "idAnonymous": df_new_mode["idAnonymous"],
        "mode": mode,
        "cluster": knn.predict(points),
        "confidence": knn.predict_proba(points).max(axis=1),
    }))
    print(f"{mode}: {len(X_train)} training, {len(points)} new replays classified")

df_knn_movement_camera = pd.concat(knn_frames, ignore_index=True)
df_knn_movement_camera = add_cluster_votes(df_knn_movement_camera, cluster_labels[METHOD])  # NEW: look up the votes of the predicted cluster
print(df_knn_movement_camera.head())

EXTRASHORT: 9495 training, 5111 new replays classified
SHORT: 9944 training, 1410 new replays classified
NORMAL: 9988 training, 655 new replays classified
ONESTACK: 9983 training, 956 new replays classified
INCLINED: 9976 training, 355 new replays classified
LONG: 9987 training, 393 new replays classified
INCLINEDSHORT: 9928 training, 870 new replays classified
   idAnonymous        mode  cluster  confidence  voters  \
0       500002  EXTRASHORT        0         1.0      27   
1       500003  EXTRASHORT        9         0.6      14   
2       500004  EXTRASHORT        5         1.0       5   
3       500005  EXTRASHORT        6         1.0      24   
4       500006  EXTRASHORT        3         1.0      35   

                                     technique_votes          suspicious_votes  
0  {'ninja': 0.89, 'sneak': 0.11, 'noshift': 0.04...             {0: 24, 1: 1}  
1  {'ninja': 0.79, 'moon': 0.07, 'unclear': 0.07,...                   {0: 13}  
2  {'noshift': 0.6, 'ninja': 0.4, 'sta

### 04b - mcplayhd concept

In [13]:
METHOD = "mcplayhd_concept"
POSITION_KEY = "reduced"

knn_frames = []
for mode, result in cluster_models[METHOD].items():
    X_train, y_train = training_data(result, POSITION_KEY)

    knn = KNeighborsClassifier(n_neighbors=KNN_NEIGHBORS)
    knn.fit(X_train, y_train)

    df_new_mode = new_features[METHOD][new_features[METHOD]["fastBuilderMode"] == mode]
    points = transform_new(result, df_new_mode)

    knn_frames.append(pd.DataFrame({
        "idAnonymous": df_new_mode["idAnonymous"],
        "mode": mode,
        "cluster": knn.predict(points),
        "confidence": knn.predict_proba(points).max(axis=1),
    }))
    print(f"{mode}: {len(X_train)} training, {len(points)} new replays classified")

df_knn_mcplayhd_concept = pd.concat(knn_frames, ignore_index=True)
df_knn_mcplayhd_concept = add_cluster_votes(df_knn_mcplayhd_concept, cluster_labels[METHOD])  # NEW: look up the votes of the predicted cluster
print(df_knn_mcplayhd_concept.head())

EXTRASHORT: 9498 training, 5100 new replays classified
SHORT: 9938 training, 1405 new replays classified
NORMAL: 9943 training, 654 new replays classified
ONESTACK: 9954 training, 951 new replays classified
INCLINED: 9907 training, 357 new replays classified
LONG: 9842 training, 388 new replays classified
INCLINEDSHORT: 9923 training, 868 new replays classified
   idAnonymous        mode  cluster  confidence  voters  \
0       500002  EXTRASHORT        0    1.000000      16   
1       500003  EXTRASHORT        8    0.533333      12   
2       500004  EXTRASHORT       17    0.733333       6   
3       500005  EXTRASHORT        9    0.666667       5   
4       500006  EXTRASHORT       12    0.666667      21   

                                     technique_votes suspicious_votes  
0  {'sneak': 0.44, 'ninja': 0.44, 'noob': 0.25, '...          {0: 16}  
1                                   {'noshift': 1.0}     {1: 2, 0: 1}  
2    {'ninja': 0.67, 'noshift': 0.17, 'telly': 0.17}     {0: 5, 1

## evaluation

Careful: the cluster labels come from k-means on the same points, so checking k-NN against them on the training data is circular.

Ideas:
- how much do the two methods agree on the new replays?
- check a few new replays per label in Minecraft (like the cluster export in 03)
- leave-one-out on the 1170 hand labels (`replay_labels.csv`)
- the blocks-per-click rule (`df_new_bpc`: bpc >= ~0.95 and speed > ~6 b/s) vs. the predicted labels

## export

In [14]:
df_knn_movement_camera.to_csv(KNN_RESULTS_PATHS["movement_camera"], index=False)
df_knn_mcplayhd_concept.to_csv(KNN_RESULTS_PATHS["mcplayhd_concept"], index=False)
print(f"exported {len(df_knn_movement_camera)} + {len(df_knn_mcplayhd_concept)} rows")

exported 9750 + 9723 rows
